# Importing Packages

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, when, to_date, trim, to_timestamp, regexp_replace, lag, sum, date_trunc
from pyspark.sql.window import Window
from pyspark.sql.types import *
load_dotenv()

# Defining Variables

In [0]:
SILVER_SCHEMA_PATH=os.getenv('SILVER_SCHEMA_PATH')
BRONZE_SCHEMA_PATH=os.getenv('BRONZE_SCHEMA_PATH')

# Reusable Functions

In [0]:
def parse_mixed_date(df, col_name, output_col=None):
    out_col = output_col if output_col else col_name

    return df.withColumn(
        out_col,
        when(
            trim(col(col_name)).rlike(r"^\d{1,2}-\d{1,2}-\d{4}$"),
            to_date(trim(col(col_name)), "dd-MM-yyyy")
        ).when(
            trim(col(col_name)).rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"),
            to_date(trim(col(col_name)),"dd/MM/yyyy")
        )
        .when(
            trim(col(col_name)).rlike(r"^\d{4}/\d{1,2}/\d{1,2}$"),
            to_date(trim(col(col_name)), "yyyy/MM/dd")
        )
        .when(
            trim(col(col_name)).rlike(r"^\d{4}-\d{1,2}-\d{1,2}$"),
            to_date(trim(col(col_name)), "yyyy-MM-dd")
        ).otherwise(None)
    )

In [0]:
def parse_mixed_datetime(df, input_col, output_col=None):
    out_col = output_col if output_col else input_col
    return df.withColumn(
        out_col,
        when(
            trim(col(input_col)).rlike(r"^\d{1,2}-\d{1,2}-\d{4}\s\d{1,2}:\d{2}$"),
            to_timestamp(trim(col(input_col)), "dd-MM-yyyy HH:mm")
        ).when(
            trim(col(input_col)).rlike(r"^\d{1,2}/\d{1,2}/\d{4}\s\d{1,2}:\d{2}$"),
            to_timestamp(trim(col(input_col)), "d/M/yyyy H:mm")
        )
        .otherwise(None)
    )

# Date Cleaning

## Opportunity Table

In [0]:
opportunity_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_opportunity`""")

### Typecasting

In [0]:
opportunity_df=parse_mixed_datetime(opportunity_df,"created_timestamp")

In [0]:
opportunity_df=opportunity_df.withColumn("opportunity_id",col("opportunity_id").cast(IntegerType()))
opportunity_df=opportunity_df.withColumn("employee_id",col("employee_id").cast(IntegerType()))
opportunity_df=opportunity_df.withColumn("customer_id",col("customer_id").cast(IntegerType()))
opportunity_df=opportunity_df.withColumn("product_id",col("product_id").cast(IntegerType()))

In [0]:
opportunity_df=parse_mixed_date(opportunity_df,"start_date")
opportunity_df=parse_mixed_date(opportunity_df,"end_date")

In [0]:
opportunity_df=opportunity_df.withColumn("start_date",col("start_date").cast(DateType()))
opportunity_df=opportunity_df.withColumn("end_date",col("end_date").cast(DateType()))

### Removing Unnecessary Symbols

In [0]:
opportunity_df=opportunity_df.withColumn(
    "revenue_amount",
    regexp_replace(col("revenue_amount"), r"[£$€,]", "").cast("double")
)

In [0]:
opportunity_df=opportunity_df.withColumn("Month",date_trunc('month',col("start_date")).cast(DateType()))

# Creating Schema

In [0]:
spark.sql(f"""create schema if not exists {SILVER_SCHEMA_PATH}""")

# Saving Dataframe

### Applying SCD Type 1 for Oppotunity Table

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(f"{SILVER_SCHEMA_PATH}.silver_opportunity"):
    dt = DeltaTable.forName(spark, f"{SILVER_SCHEMA_PATH}.silver_opportunity")
    dt.alias("target").merge(
        opportunity_df.alias("source"),
        "target.opportunity_id = source.opportunity_id" 
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    opportunity_df.write.format("delta").saveAsTable(f"{SILVER_SCHEMA_PATH}.silver_opportunity")